# Push 80 Valid Colab
Notebook ini untuk mendorong macro-F1 ke 0.78-0.80 secara valid (tanpa leakage), dengan trial model kuat + evaluasi test per trial + class multiplier tuning.

## 0) Setup Runtime
Pilih GPU T4: Runtime > Change runtime type > T4 GPU.

In [ ]:
import os
HF_TOKEN = 'hf_xxx_your_token_here'  # placeholder
os.environ['HF_TOKEN'] = HF_TOKEN
!git clone https://github.com/kzquandary/mbg-sentiment.git
%cd mbg-sentiment
!git checkout main
!python3 -m pip install -r requirements.txt
!nvidia-smi


## 1) Pilih Dataset
Default `v3_aug` (augment minority).

In [ ]:
# Toggle dataset
DATASET_PATH = 'data/dataset_relabel_mbg_improved_v3_aug.csv'  # default
# DATASET_PATH = 'data/dataset_relabel_mbg_improved_v2_boost.csv'
# DATASET_PATH = 'data/dataset_relabel_mbg_improved.csv'
assert os.path.exists(DATASET_PATH), f'Dataset tidak ditemukan: {DATASET_PATH}'
print('Dataset:', DATASET_PATH)
print('Trial pack:', 'src/resources/step7_push80_valid_trials.json')


## 2) Split + Preprocess + Internal Val + Balance (CPU)

In [ ]:
!python3 src/05_split_data.py --input "$DATASET_PATH" --text-col text --label-col Labeling_Sentimen --train-ratio 0.7 --val-ratio 0 --test-ratio 0.3
!python3 src/03_preprocess_text.py --train-input data/train.csv --test-input data/test.csv --text-col text


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv('data/train.csv')
train_sub, val_sub = train_test_split(df, test_size=0.15, random_state=42, stratify=df['Labeling_Sentimen'].astype(str))
train_sub.to_csv('data/train_sub.csv', index=False, encoding='utf-8-sig')
val_sub.to_csv('data/val_sub.csv', index=False, encoding='utf-8-sig')
print('train_sub', len(train_sub), train_sub['Labeling_Sentimen'].value_counts().to_dict())
print('val_sub  ', len(val_sub), val_sub['Labeling_Sentimen'].value_counts().to_dict())


In [ ]:
!python3 src/05b_balance_train.py --input data/train_sub.csv --label-col Labeling_Sentimen --target-by-label-json src/resources/train_balance_push80_valid.json --output data/train_sub_balanced.csv --log-output outputs/train_sub_balance_log.json


## 3) Train + Evaluate per Trial (GPU)
Setiap trial dievaluasi di test set. Hasil leaderboard disimpan otomatis.

In [ ]:
import os, json, shutil, subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'
pack = json.load(open('src/resources/step7_push80_valid_trials.json', encoding='utf-8'))
results = []
Path('outputs/figures').mkdir(parents=True, exist_ok=True)
for trial in pack:
    trial_name = trial['trial_name']
    print(f'\n=== RUN: {trial_name} ===')
    tmp_json = Path('outputs') / f'tmp_{trial_name}.json'
    tmp_json.write_text(json.dumps([trial], ensure_ascii=False, indent=2), encoding='utf-8')
    train_cmd = [
      'python3','src/07_indobert_bilstm.py',
      '--train','data/train_sub_balanced.csv',
      '--val','data/val_sub.csv',
      '--trial-configs-json',str(tmp_json),
      '--max-trials','1'
    ]
    tr = subprocess.run(train_cmd, text=True, capture_output=True)
    if tr.returncode != 0:
      print(tr.stdout)
      print(tr.stderr)
      results.append({'trial_name': trial_name, 'status':'train_failed'})
      continue
    # Tune multiplier on val
    tune_cmd = [
      'python3','src/09b_tune_class_multipliers.py',
      '--val','data/val_sub.csv',
      '--model-path','models/best_indobert_bilstm.pt',
      '--output','outputs/best_class_multipliers.json',
      '--summary-output','outputs/class_multiplier_tuning_summary.json'
    ]
    tu = subprocess.run(tune_cmd, text=True, capture_output=True)
    if tu.returncode != 0:
      print(tu.stdout)
      print(tu.stderr)
      results.append({'trial_name': trial_name, 'status':'tune_failed'})
      continue
    eval_cmd = [
      'python3','src/09_evaluate.py',
      '--test','data/test.csv',
      '--model-path','models/best_indobert_bilstm.pt',
      '--class-multiplier-json','outputs/best_class_multipliers.json'
    ]
    ev = subprocess.run(eval_cmd, text=True, capture_output=True)
    if ev.returncode != 0:
      print(ev.stdout)
      print(ev.stderr)
      results.append({'trial_name': trial_name, 'status':'eval_failed'})
      continue
    metrics = json.load(open('outputs/final_metrics.json', encoding='utf-8'))
    row = {
      'trial_name': trial_name,
      'model_name': trial['model_name'],
      'max_len': trial['max_len'],
      'batch_size': trial['batch_size'],
      'hidden_size': trial['hidden_size'],
      'loss_type': trial['loss_type'],
      'freeze_bert': trial['freeze_bert'],
      'accuracy': metrics.get('accuracy'),
      'precision_macro': metrics.get('precision_macro'),
      'recall_macro': metrics.get('recall_macro'),
      'f1_macro': metrics.get('f1_macro'),
      'status': 'ok'
    }
    results.append(row)
    # archive trial artifacts
    Path('outputs').joinpath(f'final_metrics_{trial_name}.json').write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding='utf-8')
    shutil.copy('outputs/classification_report.csv', f'outputs/classification_report_{trial_name}.csv')
    shutil.copy('outputs/confusion_matrix.csv', f'outputs/confusion_matrix_{trial_name}.csv')
    if Path('outputs/figures/confusion_matrix.png').exists():
      shutil.copy('outputs/figures/confusion_matrix.png', f'outputs/figures/confusion_matrix_{trial_name}.png')
df_res = pd.DataFrame(results)
if not df_res.empty:
    df_res = df_res.sort_values('f1_macro', ascending=False, na_position='last')
    df_res.to_csv('outputs/push80_valid_results.csv', index=False, encoding='utf-8-sig')
    display(df_res)
    print('Saved leaderboard: outputs/push80_valid_results.csv')


## 4) Tampilkan Best Trial Detail

In [ ]:
import pandas as pd
from IPython.display import display, Image
res = pd.read_csv('outputs/push80_valid_results.csv')
best = res.iloc[0]
best_trial = best['trial_name']
print('Best trial:', best_trial)
print(best[['accuracy','precision_macro','recall_macro','f1_macro']])
report = pd.read_csv(f'outputs/classification_report_{best_trial}.csv')
display(report)
img_path = f'outputs/figures/confusion_matrix_{best_trial}.png'
if os.path.exists(img_path):
    display(Image(filename=img_path))


## 5) Output
- `outputs/push80_valid_results.csv` (leaderboard trial)
- `outputs/final_metrics_<trial>.json`
- `outputs/classification_report_<trial>.csv`
- `outputs/confusion_matrix_<trial>.csv`
- `outputs/figures/confusion_matrix_<trial>.png`